In [1]:
# libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator
import time

#Visualizers
from yellowbrick.classifier import ClassificationReport
from yellowbrick.classifier import ClassPredictionError
from yellowbrick.classifier import ConfusionMatrix
from yellowbrick.classifier import ROCAUC
from yellowbrick.classifier import PrecisionRecallCurve
import matplotlib.pyplot as plt

#Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import hamming_loss
from sklearn.metrics import log_loss
from sklearn.metrics import zero_one_loss
from sklearn.metrics import matthews_corrcoef

#Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
from keras import optimizers

In [3]:

import matplotlib.pyplot as plt
import numpy as np
import joblib


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
df=pd.read_excel("/content/drive/MyDrive/PaperExtracciónDental/dataset_extraccion_up_low_bimax.xlsx")

In [6]:
df.drop(columns=['Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',"Extraccion / no extraccion inf","Extraccion / no extraccion sup", "Extraccion / no extraccion","Paciente","Extraccion tejidos blandos",'discrepancia total inferior', 'discrepancia total superior',"Clase real"],inplace=True)

In [7]:
from sklearn.preprocessing import LabelEncoder

In [8]:
LabelEncoder_1=LabelEncoder()
df["Genero"]=LabelEncoder_1.fit_transform(df["Genero"])
df["Etnia"]=LabelEncoder_1.fit_transform(df["Etnia"])
df["Clasificación  esqueletica"]=LabelEncoder_1.fit_transform(df["Clasificación  esqueletica"])
df["Relación molar"]=LabelEncoder_1.fit_transform(df["Relación molar"])
df["Relacion canina"]=LabelEncoder_1.fit_transform(df["Relacion canina"])
df["Relación premolar"]=LabelEncoder_1.fit_transform(df["Relación premolar"])


In [9]:
df = df[df['Tipo de extraccióin global'] != 'Extracción sup.']
df['Tipo de extraccióin global'] = df['Tipo de extraccióin global'].map({'Extracción bima': 1, 'No extracción': 0,"Extracción inf.":1})

In [10]:
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'],axis=1)
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [11]:
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
from sklearn import decomposition

pca = decomposition.PCA(n_components=0.96,svd_solver='full',tol=1e-4)
pca.fit(X_train)
X_train = pca.transform(X_train)
X_test=pca.transform(X_test)

In [20]:
print('Train data shape:', X_train.shape)
print('Train labels shape:', y_train.shape)
print('Test data shape:', X_test.shape)
print('Test labels shape:', y_test.shape)

Train data shape: (438, 19)
Train labels shape: (438,)
Test data shape: (98, 19)
Test labels shape: (98,)


In [21]:
#classes
classes = [0, 1]

#MLP

In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('clf', MLPClassifier())])

parameters = {
'clf__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'clf__activation': ['logistic', 'relu'],
    'clf__solver': ['adam', 'sgd'],
    'clf__learning_rate': ['constant', 'adaptive'],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=Pipeline(steps=[('clf', MLPClassifier())]),
             param_grid={'clf__activation': ['logistic', 'relu'],
                         'clf__hidden_layer_sizes': [(50,), (100,), (50, 50),
                                                     (100, 50)],
                         'clf__learning_rate': ['constant', 'adaptive'],
                         'clf__solver': ['adam', 'sgd']})

In [23]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__activation': 'relu', 'clf__hidden_layer_sizes': (100, 50), 'clf__learning_rate': 'adaptive', 'clf__solver': 'adam'}
Puntuación de validación cruzada media del mejor modelo:
0.9291797283176594


#SGD

In [24]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier


pipeline = Pipeline([    ('clf', SGDClassifier())])

parameters = {
    'clf__loss': ['hinge', 'log', 'modified_huber'],
    'clf__penalty': ['l2', 'l1', 'elasticnet'],
    'clf__alpha': [0.0001, 0.001, 0.01],
    'clf__max_iter': [1000, 2000, 3000],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Pipeline(steps=[('clf', SGDClassifier())]),
             param_grid={'clf__alpha': [0.0001, 0.001, 0.01],
                         'clf__loss': ['hinge', 'log', 'modified_huber'],
                         'clf__max_iter': [1000, 2000, 3000],
                         'clf__penalty': ['l2', 'l1', 'elasticnet']})

In [25]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__alpha': 0.01, 'clf__loss': 'hinge', 'clf__max_iter': 3000, 'clf__penalty': 'l1'}
Puntuación de validación cruzada media del mejor modelo:
0.9247126436781608


#GB

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', GradientBoostingClassifier())])
parameters = {
 'classifier__n_estimators': [50, 100, 200],  # Number of boosting stages
    'classifier__learning_rate': [0.1, 0.05, 0.01],  # Learning rate
    'classifier__max_depth': [3, 5, 7],

    'classifier__random_state':[10,20,30,40,50]
    # Maximum depth of individual regression estimators
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier',
                                        GradientBoostingClassifier())]),
             param_grid={'classifier__learning_rate': [0.1, 0.05, 0.01],
                         'classifier__max_depth': [3, 5, 7],
                         'classifier__n_estimators': [50, 100, 200],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [14]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__random_state': 30}
Puntuación de validación cruzada media del mejor modelo:
0.8948798328108672


#DTC

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('clf', DecisionTreeClassifier())])

parameters = {
    'clf__criterion': ['gini', 'entropy'],
    'clf__max_depth': [None, 5, 10, 15],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('clf', DecisionTreeClassifier())]),
             param_grid={'clf__criterion': ['gini', 'entropy'],
                         'clf__max_depth': [None, 5, 10, 15],
                         'clf__min_samples_leaf': [1, 2, 4],
                         'clf__min_samples_split': [2, 5, 10]})

In [16]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__criterion': 'entropy', 'clf__max_depth': 15, 'clf__min_samples_leaf': 4, 'clf__min_samples_split': 5}
Puntuación de validación cruzada media del mejor modelo:
0.8197230929989552


#KNN

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import Normalizer, StandardScaler, MinMaxScaler, PowerTransformer, MaxAbsScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()),

])

# Definir la cuadrícula de parámetros a explorar
param_grid = {
    'knn__n_neighbors': [1,2,3,4, 5,6, 7,10],
    'knn__weights': ['uniform', 'distance'],
    'knn__algorithm':["auto", "ball_tree", "kd_tree", "brute"],
    'scaler': [StandardScaler(), MinMaxScaler(),
 Normalizer(), MaxAbsScaler()],
 'knn__n_neighbors': [1, 3, 5, 7, 10],
 'knn__p': [1, 2],
 'knn__leaf_size': [1, 5, 10, 15],
}

grid_search =  GridSearchCV(pipeline, param_grid, cv=2).fit(X_train, y_train)


In [18]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'knn__algorithm': 'auto', 'knn__leaf_size': 1, 'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'distance', 'scaler': Normalizer()}
Puntuación de validación cruzada media del mejor modelo:
0.860730593607306


#RF

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', RandomForestClassifier())])
parameters = {
  'classifier__n_estimators': [50, 100, 150,200],
    'classifier__max_depth': [None, 10, 20,30,40],
    'classifier__min_samples_split': [2, 4, 6,8,10],
    'classifier__random_state':[10,20,30,40,50]
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier',
                                        RandomForestClassifier())]),
             param_grid={'classifier__max_depth': [None, 10, 20, 30, 40],
                         'classifier__min_samples_split': [2, 4, 6, 8, 10],
                         'classifier__n_estimators': [50, 100, 150, 200],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [20]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__max_depth': 10, 'classifier__min_samples_split': 4, 'classifier__n_estimators': 50, 'classifier__random_state': 40}
Puntuación de validación cruzada media del mejor modelo:
0.8902298850574712


#ETC

In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', ExtraTreesClassifier())])

parameters = {
    'classifier__n_estimators': [50, 100, 150,200,250,300],

    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 4, 6],
    'classifier__min_samples_split': [2,4,6,8,10],
    'classifier__random_state':[10,20,30,40,50]
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier', ExtraTreesClassifier())]),
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__min_samples_split': [2, 4, 6, 8, 10],
                         'classifier__n_estimators': [50, 100, 150, 200, 250,
                                                      300],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [22]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__max_depth': 20, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300, 'classifier__random_state': 10}
Puntuación de validación cruzada media del mejor modelo:
0.9178422152560083


#SVC